<a href="https://colab.research.google.com/github/mehmetgul/NLP/blob/main/CS7347_HW2_Solution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CS 7347 Natural Language Processing — Homework 2
### Kennesaw State University, Computer Science

---

## Question 1 — Naive Bayes Classification (25 pts)

**Given:**
| Doc | Words | Class | Set |
|-----|-------|-------|-----|
| d1 | Chinese Beijing Chinese | B | Training |
| d2 | Chinese Chinese Shanghai | B | Training |
| d3 | Tokyo Japan Chinese | A | Training |
| d4 | Chinese Macao | B | Training |
| d5 | Chinese Chinese Chinese Tokyo Japan | ? | Testing |

**Task:** Classify d5 using Naive Bayes.

In [38]:
import math
from collections import Counter

# Training Data
train = {
    'd1': (['chinese', 'beijing', 'chinese'], 'B'),
    'd2': (['chinese', 'chinese', 'shanghai'], 'B'),
    'd3': (['tokyo', 'japan', 'chinese'], 'A'),
    'd4': (['chinese', 'macao'], 'B'),
}

test_d5 = ['chinese', 'chinese', 'chinese', 'tokyo', 'japan']

# Separate by class
docs_B = {k: v for k, v in train.items() if v[1] == 'B'}
docs_A = {k: v for k, v in train.items() if v[1] == 'A'}

print("Class B docs:", list(docs_B.keys()))
print("Class A docs:", list(docs_A.keys()))

Class B docs: ['d1', 'd2', 'd4']
Class A docs: ['d3']


### Step 1 — Before Probabilities

In [39]:
N = len(train)
P_B = len(docs_B) / N
P_A = len(docs_A) / N

print(f"Total training documents: N = {N}")
print(f"P(B) = {len(docs_B)}/{N} = {P_B:.4f}")
print(f"P(A) = {len(docs_A)}/{N} = {P_A:.4f}")

Total training documents: N = 4
P(B) = 3/4 = 0.7500
P(A) = 1/4 = 0.2500


### Step 2 — Vocabulary and Word Counts per Class

In [40]:
# Collect all words for per class
words_B = [w for k in docs_B for w in train[k][0]]
words_A = [w for k in docs_A for w in train[k][0]]

count_B = Counter(words_B)
count_A = Counter(words_A)

vocab = sorted(set(words_B + words_A))
V = len(vocab)
total_B = len(words_B)
total_A = len(words_A)

print(f"Vocabulary: {vocab}")
print(f"|V| = {V}")
print(f"Total words in B: {total_B}")
print(f"Total words in A: {total_A}")
print(f"\nWord counts in B: {dict(count_B)}")
print(f"Word counts in A: {dict(count_A)}")

Vocabulary: ['beijing', 'chinese', 'japan', 'macao', 'shanghai', 'tokyo']
|V| = 6
Total words in B: 8
Total words in A: 3

Word counts in B: {'chinese': 5, 'beijing': 1, 'shanghai': 1, 'macao': 1}
Word counts in A: {'tokyo': 1, 'japan': 1, 'chinese': 1}


### Step 3 — Conditional Probabilities

$$P(w \mid c) = \frac{\text{count}(w, c) + 1}{\text{total words in } c + |V|}$$

In [41]:
import pandas as pd

like_B = {}
like_A = {}

rows = []
for w in vocab:
    cb = count_B.get(w, 0)
    ca = count_A.get(w, 0)
    like_B[w] = (cb + 1) / (total_B + V)
    like_A[w] = (ca + 1) / (total_A + V)
    rows.append({
        'Word': w,
        'count(w,B)': cb,
        'P(w|B)': f"({cb}+1)/({total_B}+{V}) = {like_B[w]:.4f}",
        'count(w,A)': ca,
        'P(w|A)': f"({ca}+1)/({total_A}+{V}) = {like_A[w]:.4f}",
    })

df_like = pd.DataFrame(rows)
print(df_like.to_string(index=False))

    Word  count(w,B)               P(w|B)  count(w,A)               P(w|A)
 beijing           1 (1+1)/(8+6) = 0.1429           0 (0+1)/(3+6) = 0.1111
 chinese           5 (5+1)/(8+6) = 0.4286           1 (1+1)/(3+6) = 0.2222
   japan           0 (0+1)/(8+6) = 0.0714           1 (1+1)/(3+6) = 0.2222
   macao           1 (1+1)/(8+6) = 0.1429           0 (0+1)/(3+6) = 0.1111
shanghai           1 (1+1)/(8+6) = 0.1429           0 (0+1)/(3+6) = 0.1111
   tokyo           0 (0+1)/(8+6) = 0.0714           1 (1+1)/(3+6) = 0.2222


### Step 4 — After Computation for d5

In [42]:
print(f"d5 = {test_d5}\n")

# Avoid underflow by using log probabilities
log_post_B = math.log(P_B)
log_post_A = math.log(P_A)

print(f"log P(B) = ln({P_B:.4f}) = {math.log(P_B):.4f}")
print(f"log P(A) = ln({P_A:.4f}) = {math.log(P_A):.4f}\n")

for w in test_d5:
    lB = math.log(like_B[w])
    lA = math.log(like_A[w])
    log_post_B += lB
    log_post_A += lA
    print(f"  log P({w}|B) = ln({like_B[w]:.4f}) = {lB:.4f}   |   log P({w}|A) = ln({like_A[w]:.4f}) = {lA:.4f}")

print(f"\n{'='*60}")
print(f"log P(B|d5) ∝ {log_post_B:.4f}")
print(f"log P(A|d5) ∝ {log_post_A:.4f}")
print(f"\n==> Since {log_post_B:.4f} > {log_post_A:.4f}, d5 is classified as: B")

d5 = ['chinese', 'chinese', 'chinese', 'tokyo', 'japan']

log P(B) = ln(0.7500) = -0.2877
log P(A) = ln(0.2500) = -1.3863

  log P(chinese|B) = ln(0.4286) = -0.8473   |   log P(chinese|A) = ln(0.2222) = -1.5041
  log P(chinese|B) = ln(0.4286) = -0.8473   |   log P(chinese|A) = ln(0.2222) = -1.5041
  log P(chinese|B) = ln(0.4286) = -0.8473   |   log P(chinese|A) = ln(0.2222) = -1.5041
  log P(tokyo|B) = ln(0.0714) = -2.6391   |   log P(tokyo|A) = ln(0.2222) = -1.5041
  log P(japan|B) = ln(0.0714) = -2.6391   |   log P(japan|A) = ln(0.2222) = -1.5041

log P(B|d5) ∝ -8.1077
log P(A|d5) ∝ -8.9067

==> Since -8.1077 > -8.9067, d5 is classified as: B


---
## Question 2 — Cross-Validation (15 pts)

### What is Cross-Validation?

Cross-validation is a statistical method of testing the capability of machine learning model to generalize on unknown data. As opposed to the one train/test split, the data is divided into **k** equal folds, and the model is trained and assessed 1/k times - each time a different fold is used as the test set and the rest of the folds as the training set. The averages are used to obtain more credible estimate of performance.

Concrete: 5-Fold Cross-Validation on a Spam Classifier.

Considering the scenario of having two categories: **1,000 emails in the spam and not-spam category and wish to test a Naive Bayes classifier.

**Steps:**

1. **Shuffle** the 1,000 emails randomly.
2. **Divide** into k = 5 equal folds (200 emails in each of them): Fold 1, Fold 2, etc., Fold 5.
3. **Iterate** through 5 rounds:
   - **Round 1: Train on Folds 2-5 (800 emails  of them), Test on Fold 1 (200 emails) Accuracy 1.
   - **Round 2**: Fold 1, 3-5: Train on Fold 1, test on Fold 2, Accuracy 2.
   - Round 3: Train Folds 1245 test Fold 3 Accuracy 3
   - **Round 4**: Fold 153 training on Folds 1-3, 5, Folds 4 testing= Accuracy 4.
   - **Round 5**: Train on Folds 14- Train on Fold 5 Test on Fold 14 Accuracy 5.
4. **Average**, Final Accuracy = (Accuracy1 + Accuracy2 +...+ Accuracy 5)/5.

### When to Use Cross-Validation

- A small dataset requires one split to give an inaccurate estimate.
- At the stage of the **model selection and hyperparameter optimization e.g. the optimal regularization strength, number of layers, or kernel - prior to final evaluation on a held-out test set.
- When using several models in order to decide which generalizes best.
- It is applicable at the development/validation stage but not on the final test set to be used in reporting results.

In [43]:
# 5-Fold Cross-Validation splits

folds = [f"Fold {i+1}" for i in range(5)]

print("5-Fold Cross-Validation Splits:")
print("="*65)
for i in range(5):
    test = folds[i]
    train_folds = [f for j, f in enumerate(folds) if j != i]
    print(f"Round {i+1}:  Train = {train_folds}  |  Test = [{test}]")
print("="*65)
print("Final metric = average of all 5 rounds")

5-Fold Cross-Validation Splits:
Round 1:  Train = ['Fold 2', 'Fold 3', 'Fold 4', 'Fold 5']  |  Test = [Fold 1]
Round 2:  Train = ['Fold 1', 'Fold 3', 'Fold 4', 'Fold 5']  |  Test = [Fold 2]
Round 3:  Train = ['Fold 1', 'Fold 2', 'Fold 4', 'Fold 5']  |  Test = [Fold 3]
Round 4:  Train = ['Fold 1', 'Fold 2', 'Fold 3', 'Fold 5']  |  Test = [Fold 4]
Round 5:  Train = ['Fold 1', 'Fold 2', 'Fold 3', 'Fold 4']  |  Test = [Fold 5]
Final metric = average of all 5 rounds


---
## Question 3 — Gradient Descent (15 pts)

### How Gradient Descent Works

The gradient descent is an iterative optimization procedure that minimizes a loss function f (x, w) by continuously updating the parameters w in the negative direction of the gradient:

$$w_{t+1} = w_t - \eta \frac{\partial}{\partial w} f(x, w)$$

**Algorithm Steps:**
1. Initialize weights $w_0$ (randomly or to zeros).
2. Derive the gradient $\frac{\partial}{\partial w} f(x, w)$ — this tells us the direction of steepest *increase*.
3. Update: $w_{t+1} = w_t - \eta \cdot \text{gradient}$ — we move *opposite* to the gradient.
4. We repeat until convergence (gradient ≈ 0 or loss change is small).

### Effect of Learning Rate $\eta$

| $\eta$ | Effect |
|--------|--------|
| **Too small** | Very slow convergence; may take thousands of extra iterations. |
| **Too large** | Overshoots the minimum; loss oscillates or diverges. |
| **Just right** | Steady, efficient convergence to the minimum. |

### Comparison of Gradient Descent Variants

| Variant | Data per Update | Speed per Iteration | Update Stability | Memory |
|---------|----------------|---------------------|-----------------|--------|
| **Batch GD** | Entire dataset | Slow (full pass) | Very smooth/stable | High |
| **Stochastic GD (SGD)** | 1 random sample | Very fast | Noisy/erratic | Low |
| **Mini-Batch GD** | Small batch (e.g. 32) | Fast & GPU-friendly | Good balance | Moderate |

- **Batch GD**: Exact gradient → smooth convergence but expensive per step.
- **SGD**: Noisy gradient from one sample → fast but erratic; noise can help escape local minima.
- **Mini-Batch GD**: Most widely used in practice. Balances noise and stability; leverages GPU parallelism.

In [44]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Demonstration: effect of learning rate on gradient descent
# Minimize f(w) = w^2, gradient = 2w

def run_gd(lr, w_init=5.0, steps=20):
    ws = [w_init]
    for _ in range(steps):
        grad = 2 * ws[-1]       # df/dw = 2w
        w_new = ws[-1] - lr * grad
        ws.append(w_new)
    return ws

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
lrs = [0.01, 0.3, 0.55]
titles = ['η = 0.01 (too small)', 'η = 0.3 (good)', 'η = 0.55 (too large)']

for ax, lr, title in zip(axes, lrs, titles):
    ws = run_gd(lr)
    losses = [w**2 for w in ws]
    ax.plot(losses, 'o-', markersize=4)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Loss f(w) = w²')
    ax.set_ylim(-1, 30)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('learning_rate_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print("Plot saved: learning_rate_comparison.png")

Plot saved: learning_rate_comparison.png


---
## Question 4 — TF-IDF, Co-occurrence, Cosine Similarity (45 pts)

### Given Text Documents:
1. It is going to rain today.
2. Today I am not going outside.
3. NLP is an interesting topic.
4. NLP includes ML, DL topics too.
5. I am going to complete NLP homework, today.

### Part (a) — TF-IDF Vectors (15 pts)

In [45]:
import re
import math
import pandas as pd
import numpy as np

# ── Step 1: Tokenization (lowercase, remove punctuation) ──
raw_docs = [
    "It is going to rain today.",
    "Today I am not going outside.",
    "NLP is an interesting topic.",
    "NLP includes ML, DL topics too.",
    "I am going to complete NLP homework, today.",
]

def tokenize(text):
    return re.findall(r'[a-z]+', text.lower())

docs = [tokenize(d) for d in raw_docs]

print("Tokenized Documents:")
for i, d in enumerate(docs):
    print(f"  D{i+1}: {d}")

Tokenized Documents:
  D1: ['it', 'is', 'going', 'to', 'rain', 'today']
  D2: ['today', 'i', 'am', 'not', 'going', 'outside']
  D3: ['nlp', 'is', 'an', 'interesting', 'topic']
  D4: ['nlp', 'includes', 'ml', 'dl', 'topics', 'too']
  D5: ['i', 'am', 'going', 'to', 'complete', 'nlp', 'homework', 'today']


In [46]:
# ── Step 2: Vocabulary Construction ──
vocab = sorted(set(token for doc in docs for token in doc))
N = len(docs)

print(f"Vocabulary ({len(vocab)} unique tokens):")
print(vocab)

Vocabulary (21 unique tokens):
['am', 'an', 'complete', 'dl', 'going', 'homework', 'i', 'includes', 'interesting', 'is', 'it', 'ml', 'nlp', 'not', 'outside', 'rain', 'to', 'today', 'too', 'topic', 'topics']


In [47]:
# ── Step 3: Term Frequency (TF) ──
# TF(t, d) = count(t in d) / |d|

from collections import Counter

tf_matrix = pd.DataFrame(0.0, index=vocab, columns=[f'D{i+1}' for i in range(N)])

for di, doc in enumerate(docs):
    counts = Counter(doc)
    doc_len = len(doc)
    for token, count in counts.items():
        tf_matrix.loc[token, f'D{di+1}'] = round(count / doc_len, 4)

print("Term Frequency (TF) Matrix:")
# Show only non-zero rows per column to save space, but display full matrix
print(tf_matrix.to_string())

Term Frequency (TF) Matrix:
                 D1      D2   D3      D4     D5
am           0.0000  0.1667  0.0  0.0000  0.125
an           0.0000  0.0000  0.2  0.0000  0.000
complete     0.0000  0.0000  0.0  0.0000  0.125
dl           0.0000  0.0000  0.0  0.1667  0.000
going        0.1667  0.1667  0.0  0.0000  0.125
homework     0.0000  0.0000  0.0  0.0000  0.125
i            0.0000  0.1667  0.0  0.0000  0.125
includes     0.0000  0.0000  0.0  0.1667  0.000
interesting  0.0000  0.0000  0.2  0.0000  0.000
is           0.1667  0.0000  0.2  0.0000  0.000
it           0.1667  0.0000  0.0  0.0000  0.000
ml           0.0000  0.0000  0.0  0.1667  0.000
nlp          0.0000  0.0000  0.2  0.1667  0.125
not          0.0000  0.1667  0.0  0.0000  0.000
outside      0.0000  0.1667  0.0  0.0000  0.000
rain         0.1667  0.0000  0.0  0.0000  0.000
to           0.1667  0.0000  0.0  0.0000  0.125
today        0.1667  0.1667  0.0  0.0000  0.125
too          0.0000  0.0000  0.0  0.1667  0.000
topic       

In [48]:
# ── Step 4: Document Frequency (df) and Inverse Document Frequency (IDF) ──
# IDF(t) = log10(N / df(t))

df_counts = {}
for t in vocab:
    df_counts[t] = sum(1 for doc in docs if t in doc)

idf_values = {}
for t in vocab:
    idf_values[t] = math.log10(N / df_counts[t])

idf_df = pd.DataFrame({
    'Token': vocab,
    'df(t)': [df_counts[t] for t in vocab],
    'IDF = log10(5/df)': [round(idf_values[t], 4) for t in vocab]
}).set_index('Token')

print("IDF Values:")
print(idf_df.to_string())

IDF Values:
             df(t)  IDF = log10(5/df)
Token                                
am               2             0.3979
an               1             0.6990
complete         1             0.6990
dl               1             0.6990
going            3             0.2218
homework         1             0.6990
i                2             0.3979
includes         1             0.6990
interesting      1             0.6990
is               2             0.3979
it               1             0.6990
ml               1             0.6990
nlp              3             0.2218
not              1             0.6990
outside          1             0.6990
rain             1             0.6990
to               2             0.3979
today            3             0.2218
too              1             0.6990
topic            1             0.6990
topics           1             0.6990


In [49]:
# ── Step 5: TF-IDF = TF × IDF ──

tfidf_matrix = tf_matrix.copy()
for t in vocab:
    tfidf_matrix.loc[t] = tf_matrix.loc[t] * idf_values[t]

# Round for display
tfidf_display = tfidf_matrix.round(4)

print("TF-IDF Matrix:")
print(tfidf_display.to_string())
print()
print("Each COLUMN is the TF-IDF vector for a document.")
print("Each ROW is the TF-IDF representation of a token across documents.")

TF-IDF Matrix:
                 D1      D2      D3      D4      D5
am           0.0000  0.0663  0.0000  0.0000  0.0497
an           0.0000  0.0000  0.1398  0.0000  0.0000
complete     0.0000  0.0000  0.0000  0.0000  0.0874
dl           0.0000  0.0000  0.0000  0.1165  0.0000
going        0.0370  0.0370  0.0000  0.0000  0.0277
homework     0.0000  0.0000  0.0000  0.0000  0.0874
i            0.0000  0.0663  0.0000  0.0000  0.0497
includes     0.0000  0.0000  0.0000  0.1165  0.0000
interesting  0.0000  0.0000  0.1398  0.0000  0.0000
is           0.0663  0.0000  0.0796  0.0000  0.0000
it           0.1165  0.0000  0.0000  0.0000  0.0000
ml           0.0000  0.0000  0.0000  0.1165  0.0000
nlp          0.0000  0.0000  0.0444  0.0370  0.0277
not          0.0000  0.1165  0.0000  0.0000  0.0000
outside      0.0000  0.1165  0.0000  0.0000  0.0000
rain         0.1165  0.0000  0.0000  0.0000  0.0000
to           0.0663  0.0000  0.0000  0.0000  0.0497
today        0.0370  0.0370  0.0000  0.0000  0.02

### Part (b) — Term-Term Co-occurrence Matrix (window ± 3) (15 pts)

Each token at position i in a document is counted as co-occurring with all tokens at position *i−3* through *i+3* (not counting itself). The counts are added together in all documents.

In [50]:
from collections import defaultdict

# ── Build Co-occurrence Matrix with window = ±3 ──
cooc = defaultdict(int)

for doc in docs:
    for i, word in enumerate(doc):
        # Context window: positions i-3 to i+3 (excluding i)
        for j in range(max(0, i - 3), min(len(doc), i + 4)):
            if j != i:
                cooc[(word, doc[j])] += 1

# Build the matrix
cooc_matrix = pd.DataFrame(0, index=vocab, columns=vocab)
for (w1, w2), count in cooc.items():
    cooc_matrix.loc[w1, w2] = count

print("Term-Term Co-occurrence Matrix (window = ±3):")
print(cooc_matrix.to_string())

Term-Term Co-occurrence Matrix (window = ±3):
             am  an  complete  dl  going  homework  i  includes  interesting  is  it  ml  nlp  not  outside  rain  to  today  too  topic  topics
am            0   0         1   0      2         0  2         0            0   0   0   0    0    1        1     0   1      1    0      0       0
an            0   0         0   0      0         0  0         0            1   1   0   0    1    0        0     0   0      0    0      1       0
complete      1   0         0   0      1         1  0         0            0   0   0   0    1    0        0     0   1      1    0      0       0
dl            0   0         0   0      0         0  0         1            0   0   0   1    1    0        0     0   0      0    1      0       1
going         2   0         1   0      0         0  2         0            0   1   1   0    1    1        1     1   2      1    0      0       0
homework      0   0         1   0      0         0  0         0            0   0   0

In [51]:
# ── Show detailed co-occurrence calculation for D1 as example ──
print("Detailed Example — D1:", docs[0])
print()
for i, word in enumerate(docs[0]):
    start = max(0, i - 3)
    end = min(len(docs[0]), i + 4)
    context = [(j, docs[0][j]) for j in range(start, end) if j != i]
    print(f"  Position {i} ('{word}'): context window → {context}")

Detailed Example — D1: ['it', 'is', 'going', 'to', 'rain', 'today']

  Position 0 ('it'): context window → [(1, 'is'), (2, 'going'), (3, 'to')]
  Position 1 ('is'): context window → [(0, 'it'), (2, 'going'), (3, 'to'), (4, 'rain')]
  Position 2 ('going'): context window → [(0, 'it'), (1, 'is'), (3, 'to'), (4, 'rain'), (5, 'today')]
  Position 3 ('to'): context window → [(0, 'it'), (1, 'is'), (2, 'going'), (4, 'rain'), (5, 'today')]
  Position 4 ('rain'): context window → [(1, 'is'), (2, 'going'), (3, 'to'), (5, 'today')]
  Position 5 ('today'): context window → [(2, 'going'), (3, 'to'), (4, 'rain')]


### Part (c) — Most Similar Word Pair using Cosine Similarity (15 pts)

$$\text{cosine\_sim}(\vec{a}, \vec{b}) = \frac{\vec{a} \cdot \vec{b}}{\|\vec{a}\| \cdot \|\vec{b}\|}$$

To calculate the similarity of cosines between all pairs of words, we use the following formula:
1. TF-IDF row vectors (each word's TF-IDF values across documents)
2. Co-occurrence row vectors

In [52]:
from itertools import combinations

def cosine_similarity(vec_a, vec_b):
    """Compute cosine similarity between two vectors."""
    a = np.array(vec_a, dtype=float)
    b = np.array(vec_b, dtype=float)
    dot = np.dot(a, b)
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)

# ── (i) TF-IDF based cosine similarity ──
print("=" * 70)
print("(i) TF-IDF Based Cosine Similarity")
print("=" * 70)
print()
print("Each word's vector = its ROW in the TF-IDF matrix (values across D1..D5)")
print()

# Filter to words that have at least one non-zero TF-IDF value
active_words = [w for w in vocab if tfidf_matrix.loc[w].sum() > 0]
# Actually all words appear at least once, so all have non-zero TF-IDF

best_sim_tfidf = -1
best_pair_tfidf = None
results_tfidf = []

for w1, w2 in combinations(vocab, 2):
    vec1 = tfidf_matrix.loc[w1].values
    vec2 = tfidf_matrix.loc[w2].values
    # Skip if either vector is all zeros
    if np.linalg.norm(vec1) == 0 or np.linalg.norm(vec2) == 0:
        continue
    sim = cosine_similarity(vec1, vec2)
    results_tfidf.append((w1, w2, sim))
    if sim > best_sim_tfidf:
        best_sim_tfidf = sim
        best_pair_tfidf = (w1, w2)

# Sort by similarity descending, show top 10
results_tfidf.sort(key=lambda x: x[2], reverse=True)
print("Top 10 most similar pairs (TF-IDF):")
print(f"{'Word 1':<15} {'Word 2':<15} {'Cosine Sim':>10}")
print("-" * 42)
for w1, w2, sim in results_tfidf[:10]:
    print(f"{w1:<15} {w2:<15} {sim:>10.4f}")

print(f"\n>>> MOST SIMILAR PAIR (TF-IDF): ('{best_pair_tfidf[0]}', '{best_pair_tfidf[1]}') with cosine similarity = {best_sim_tfidf:.4f}")

(i) TF-IDF Based Cosine Similarity

Each word's vector = its ROW in the TF-IDF matrix (values across D1..D5)

Top 10 most similar pairs (TF-IDF):
Word 1          Word 2          Cosine Sim
------------------------------------------
am              i                   1.0000
an              interesting         1.0000
an              topic               1.0000
complete        homework            1.0000
dl              includes            1.0000
dl              ml                  1.0000
dl              too                 1.0000
dl              topics              1.0000
includes        ml                  1.0000
includes        too                 1.0000

>>> MOST SIMILAR PAIR (TF-IDF): ('am', 'i') with cosine similarity = 1.0000


In [53]:
# ── Show detailed calculation for the best TF-IDF pair ──
w1, w2 = best_pair_tfidf
vec1 = tfidf_matrix.loc[w1].values
vec2 = tfidf_matrix.loc[w2].values

print(f"Detailed Calculation for ('{w1}', '{w2}'):")
print(f"  Vector for '{w1}': {np.round(vec1, 4)}")
print(f"  Vector for '{w2}': {np.round(vec2, 4)}")
print(f"  Dot product: {np.dot(vec1, vec2):.6f}")
print(f"  ||{w1}|| = {np.linalg.norm(vec1):.6f}")
print(f"  ||{w2}|| = {np.linalg.norm(vec2):.6f}")
print(f"  Cosine similarity = {np.dot(vec1, vec2):.6f} / ({np.linalg.norm(vec1):.6f} × {np.linalg.norm(vec2):.6f})")
print(f"                    = {best_sim_tfidf:.6f}")

Detailed Calculation for ('am', 'i'):
  Vector for 'am': [0.     0.0663 0.     0.     0.0497]
  Vector for 'i': [0.     0.0663 0.     0.     0.0497]
  Dot product: 0.006875
  ||am|| = 0.082915
  ||i|| = 0.082915
  Cosine similarity = 0.006875 / (0.082915 × 0.082915)
                    = 1.000000


In [54]:
# ── (ii) Co-occurrence based cosine similarity ──
print("=" * 70)
print("(ii) Co-occurrence Based Cosine Similarity")
print("=" * 70)
print()
print("Each word's vector = its ROW in the co-occurrence matrix")
print()

best_sim_cooc = -1
best_pair_cooc = None
results_cooc = []

for w1, w2 in combinations(vocab, 2):
    vec1 = cooc_matrix.loc[w1].values.astype(float)
    vec2 = cooc_matrix.loc[w2].values.astype(float)
    if np.linalg.norm(vec1) == 0 or np.linalg.norm(vec2) == 0:
        continue
    sim = cosine_similarity(vec1, vec2)
    results_cooc.append((w1, w2, sim))
    if sim > best_sim_cooc:
        best_sim_cooc = sim
        best_pair_cooc = (w1, w2)

results_cooc.sort(key=lambda x: x[2], reverse=True)
print("Top 10 most similar pairs (Co-occurrence):")
print(f"{'Word 1':<15} {'Word 2':<15} {'Cosine Sim':>10}")
print("-" * 42)
for w1, w2, sim in results_cooc[:10]:
    print(f"{w1:<15} {w2:<15} {sim:>10.4f}")

print(f"\n>>> MOST SIMILAR PAIR (Co-occurrence): ('{best_pair_cooc[0]}', '{best_pair_cooc[1]}') with cosine similarity = {best_sim_cooc:.4f}")

(ii) Co-occurrence Based Cosine Similarity

Each word's vector = its ROW in the co-occurrence matrix

Top 10 most similar pairs (Co-occurrence):
Word 1          Word 2          Cosine Sim
------------------------------------------
i               outside             0.8704
includes        too                 0.8660
it              rain                0.8660
dl              ml                  0.8000
an              interesting         0.7500
going           today               0.7454
am              not                 0.7442
to              today               0.7396
complete        i                   0.7385
complete        today               0.6804

>>> MOST SIMILAR PAIR (Co-occurrence): ('i', 'outside') with cosine similarity = 0.8704


In [55]:
# ── Show detailed calculation for the best co-occurrence pair ──
w1, w2 = best_pair_cooc
vec1 = cooc_matrix.loc[w1].values.astype(float)
vec2 = cooc_matrix.loc[w2].values.astype(float)

print(f"Detailed Calculation for ('{w1}', '{w2}'):")
print(f"  Vector for '{w1}': {vec1.astype(int)}")
print(f"  Vector for '{w2}': {vec2.astype(int)}")
print(f"  Dot product: {np.dot(vec1, vec2):.4f}")
print(f"  ||{w1}|| = {np.linalg.norm(vec1):.4f}")
print(f"  ||{w2}|| = {np.linalg.norm(vec2):.4f}")
print(f"  Cosine similarity = {np.dot(vec1, vec2):.4f} / ({np.linalg.norm(vec1):.4f} × {np.linalg.norm(vec2):.4f})")
print(f"                    = {best_sim_cooc:.6f}")

Detailed Calculation for ('i', 'outside'):
  Vector for 'i': [2 0 0 0 2 0 0 0 0 0 0 0 0 1 0 0 1 1 0 0 0]
  Vector for 'outside': [1 0 0 0 1 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0]
  Dot product: 5.0000
  ||i|| = 3.3166
  ||outside|| = 1.7321
  Cosine similarity = 5.0000 / (3.3166 × 1.7321)
                    = 0.870388


---
## Summary of Results

| Question | Answer |
|----------|--------|
| Q1 — Naïve Bayes | d5 is classified as **B** |
| Q2 — Cross-Validation | k-fold technique for reliable model evaluation; use during development/tuning |
| Q3 — Gradient Descent | Iterative optimization; η controls step size; Batch vs SGD vs Mini-Batch |
| Q4a — TF-IDF | Computed TF-IDF vectors for all tokens across 5 documents |
| Q4b — Co-occurrence | Built term-term co-occurrence matrix with ±3 window |
| Q4c — Cosine Similarity | Identified most similar word pairs under both representations |